# Titanic ML Pipeline — Leakage Fixed

This is a corrected version of the demo pipeline. The original had a data leakage bug in the missing value imputation step. This notebook keeps the same overall structure but fixes that issue.

## Data Exploration

Loading both files and getting a feel for the data — types, missing values, distributions.

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

train_df.head()

In [ ]:
train_df.info()
train_df.describe(include='all')

## Feature Engineering

Extracting `Title` from the name field and computing `FamilySize` / `IsAlone`. These are all row-level operations so it's safe to do them on the combined dataset — no cross-row statistics are computed here.

In [ ]:
# combine for row-level feature extraction only
test_df['Survived'] = np.nan
full_df = pd.concat([train_df, test_df], sort=False)

full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
full_df['Title'] = full_df['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
full_df['Title'] = full_df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1
full_df['IsAlone'] = (full_df['FamilySize'] == 1).astype(int)

full_df = full_df.drop(['Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1)

# split back before any statistics are computed
train_df = full_df[full_df['Survived'].notnull()].copy()
test_df  = full_df[full_df['Survived'].isnull()].drop('Survived', axis=1).copy()

## Data Cleaning — Fixing the Leakage

**What was wrong in the original pipeline:**

The original code filled missing values like this:

```python
full_df['Embarked'] = full_df['Embarked'].fillna(full_df['Embarked'].mode()[0])
full_df['Fare']     = full_df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full_df['Age']      = full_df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
```

The problem is that `full_df` contains both training and test rows. So the mode of `Embarked`, the per-class median of `Fare`, and the per-group median of `Age` are all computed using test set passengers' values. That means test data influenced the fill values that got applied to the training set — which is leakage.

In a real deployment you won't have access to the test data when preprocessing training data. Using it here makes the imputed values slightly more accurate than they should be, which inflates performance estimates.

**The fix:**

Compute all fill statistics from the training set only, then apply those same values to both train and test.

In [ ]:
# compute fill statistics from training data only
embarked_mode  = train_df['Embarked'].mode()[0]
fare_medians   = train_df.groupby('Pclass')['Fare'].median()
age_medians    = train_df.groupby(['Title', 'Pclass'])['Age'].median()
age_global_med = train_df['Age'].median()  # fallback for unseen Title+Pclass combos

# apply to train
train_df['Embarked'] = train_df['Embarked'].fillna(embarked_mode)
train_df['Fare']     = train_df.apply(
    lambda r: fare_medians[r['Pclass']] if pd.isna(r['Fare']) else r['Fare'], axis=1
)
train_df['Age'] = train_df.apply(
    lambda r: age_medians.get((r['Title'], r['Pclass']), age_global_med) if pd.isna(r['Age']) else r['Age'], axis=1
)

# apply the same train-derived values to test
test_df['Embarked'] = test_df['Embarked'].fillna(embarked_mode)
test_df['Fare']     = test_df.apply(
    lambda r: fare_medians[r['Pclass']] if pd.isna(r['Fare']) else r['Fare'], axis=1
)
test_df['Age'] = test_df.apply(
    lambda r: age_medians.get((r['Title'], r['Pclass']), age_global_med) if pd.isna(r['Age']) else r['Age'], axis=1
)

print('Missing values after cleaning:')
print('train:', train_df[['Age', 'Fare', 'Embarked']].isnull().sum().to_dict())
print('test: ', test_df[['Age', 'Fare', 'Embarked']].isnull().sum().to_dict())

## Preprocessing

Numeric features are imputed (catch any remaining NaNs) and scaled. Categorical features are one-hot encoded. Both the imputer and the scaler are fit on training data only and then applied to test — same correct pattern as the original demo.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from numpy import hstack

X = train_df.drop('Survived', axis=1)
y = train_df['Survived'].astype(int)

categorical_cols = ['Sex', 'Embarked', 'Title']
numeric_cols     = ['Age', 'Fare', 'FamilySize', 'IsAlone', 'Pclass', 'SibSp', 'Parch']

# fit on train, transform both
num_imputer = SimpleImputer(strategy='median')
X_numeric    = num_imputer.fit_transform(X[numeric_cols])
test_numeric = num_imputer.transform(test_df[numeric_cols])

scaler       = StandardScaler()
X_numeric    = scaler.fit_transform(X_numeric)
test_numeric = scaler.transform(test_numeric)

encoder          = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_categorical    = encoder.fit_transform(X[categorical_cols])
test_categorical = encoder.transform(test_df[categorical_cols])

X_final    = hstack([X_numeric, X_categorical])
test_final = hstack([test_numeric, test_categorical])

## Model Training & Evaluation

Same Random Forest setup as the demo — 5-fold cross-validation to evaluate generalization.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

cv_scores = cross_val_score(clf, X_final, y, cv=5, scoring='accuracy')
print(f'Cross-validated accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

clf.fit(X_final, y)

## Predictions & Submission

In [ ]:
predictions = clf.predict(test_final)

submission = pd.DataFrame({
    'PassengerId': test_df.index + 892,
    'Survived': predictions.astype(int)
})
submission.to_csv('titanic_submission.csv', index=False)
print('Submission saved.')